# DDPO Baseline — Colab A100 Training Notebook

This notebook runs the full DDPO training pipeline on Google Colab Pro (A100 40GB).

**What it does:**
1. Mounts Google Drive for persistent storage
2. Verifies A100 GPU allocation
3. Pre-downloads all model weights to Drive cache (survives session restarts)
4. Clones the DDPO repo and installs dependencies
5. Launches training with auto-resume (picks up where it left off if session dies)

**All outputs (checkpoints, metrics, plots) are saved to Google Drive**, not local disk.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = "/content/drive/MyDrive/ddpo_baseline"
HF_CACHE = "/content/drive/MyDrive/hf_cache"
RUN_DIR = os.path.join(DRIVE_BASE, "run1")

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

print(f"Drive base: {DRIVE_BASE}")
print(f"HF cache:   {HF_CACHE}")
print(f"Run dir:    {RUN_DIR}")

## 2. Verify GPU — must be A100

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()

print(f"\nDetected GPU: {gpu_name}")

if "A100" in gpu_name:
    print("A100 confirmed — proceeding with bfloat16 config.")
elif "V100" in gpu_name or "T4" in gpu_name or "L4" in gpu_name:
    print(f"\n*** WARNING: Got {gpu_name} instead of A100. ***")
    print("The baseline config assumes A100 40GB memory and native bf16.")
    print("Recommendation: Runtime → Change runtime type → A100 GPU, then restart.")
    print("If you want to proceed anyway, switch config to t4_smoketest.yaml.")
else:
    print(f"\nUnrecognised GPU: {gpu_name}. Proceed with caution.")

## 3. Set HuggingFace cache to Drive (before any HF import)

In [ ]:
import os

os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE, "transformers")

print(f"HF_HOME={os.environ['HF_HOME']}")
print(f"TRANSFORMERS_CACHE={os.environ['TRANSFORMERS_CACHE']}")
print("All model weights will be cached on Drive.")

## 4. Clone repo & install dependencies

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/DDPO.git"  # ← EDIT THIS
REPO_DIR = "/content/DDPO"

if os.path.isdir(REPO_DIR):
    print(f"Repo already cloned at {REPO_DIR}, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}

print("\nInstalling dependencies...")
!pip install -q -r {REPO_DIR}/requirements.txt
print("Done.")

## 5. Pre-download model weights to Drive cache

This cell downloads ~8 GB of model weights on first run.
Subsequent sessions skip this because the weights are cached on Drive.

In [ ]:
print("Pre-downloading model weights (cached on Drive, only slow on first run)...")
print()

from huggingface_hub import snapshot_download

models_to_cache = [
    ("runwayml/stable-diffusion-v1-5", "SD 1.5 (~4 GB)"),
    ("yuvalkirstain/PickScore_v1", "PickScore (~3 GB)"),
    ("laion/CLIP-ViT-H-14-laion2B-s32B-b79K", "CLIP-H processor (~500 MB)"),
]

for model_id, desc in models_to_cache:
    print(f"  {desc}: {model_id} ... ", end="", flush=True)
    snapshot_download(model_id)
    print("OK")

print("\nLPIPS weights will be auto-downloaded on first use (~30 MB).")
print("All weights cached at:", HF_CACHE)

## 6. Check for existing checkpoints

In [ ]:
ckpt_dir = os.path.join(RUN_DIR, "checkpoints")

if os.path.isdir(ckpt_dir):
    ckpts = sorted([d for d in os.listdir(ckpt_dir) if d.startswith("step_")])
    if ckpts:
        latest = ckpts[-1]
        print(f"Found {len(ckpts)} checkpoint(s). Latest: {latest}")
        print(f"Training will AUTO-RESUME from {latest}.")
    else:
        print("Checkpoint dir exists but is empty. Starting fresh.")
else:
    print("No checkpoints found. Starting fresh training run.")

## 7. Launch training (auto-resumes if checkpoint exists)

In [ ]:
!cd {REPO_DIR} && python train_ddpo.py \
    --config configs/baseline_a100.yaml \
    --output_dir {RUN_DIR} \
    --resume_from auto

## 8. Quick Results Check

Run this after training completes (or after a partial run) to inspect progress.

In [ ]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

metrics_path = os.path.join(RUN_DIR, "metrics_history.json")

# Also check inside checkpoints for the latest partial history
if not os.path.exists(metrics_path):
    ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
    if os.path.isdir(ckpt_dir):
        ckpts = sorted([d for d in os.listdir(ckpt_dir) if d.startswith("step_")])
        if ckpts:
            metrics_path = os.path.join(ckpt_dir, ckpts[-1], "metrics_history.json")

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        history = json.load(f)
    
    steps = [m["step"] for m in history]
    pick = [m.get("eval/overall/pickscore", None) for m in history]
    div = [m.get("eval/overall/diversity", None) for m in history]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    valid = [(s, p) for s, p in zip(steps, pick) if p is not None]
    if valid:
        ax1.plot(*zip(*valid), "b-o")
        ax1.set_xlabel("Step")
        ax1.set_ylabel("PickScore")
        ax1.set_title("PickScore (higher = better)")
    
    valid = [(s, d) for s, d in zip(steps, div) if d is not None]
    if valid:
        ax2.plot(*zip(*valid), "r-o")
        ax2.set_xlabel("Step")
        ax2.set_ylabel("LPIPS Diversity")
        ax2.set_title("Diversity (lower = more collapse)")
    
    fig.tight_layout()
    plt.show()
    
    print(f"\nLoaded {len(history)} eval points, latest at step {steps[-1]}")
else:
    print("No metrics found yet. Run training first.")

In [ ]:
# Show before/after grid if it exists
grid_path = os.path.join(RUN_DIR, "before_after_grid.png")
if os.path.exists(grid_path):
    print("Before/After Comparison Grid:")
    display(Image(filename=grid_path))
else:
    print("Grid not generated yet (created at end of full training run).")